# 1 — Refresh Transactions

Fetches order data from the SBIS API for a given date range and writes to the `transactions` Delta table.

**Usage:** Set the date range in the config cell below, then Run All.

In [ ]:
# ── Date Range Config ──────────────────────────────────────────────
# Edit these dates before running. Uses today by default.

from datetime import datetime

today = datetime.now()
START_DATE = datetime(today.year, today.month, today.day, 0, 0, 0)
END_DATE   = datetime(today.year, today.month, today.day, 23, 59, 59)

# Example: custom range
# START_DATE = datetime(2025, 3, 1, 0, 0, 0)
# END_DATE   = datetime(2025, 3, 31, 23, 59, 59)

DATE_FROM = START_DATE.strftime("%Y-%m-%d")
DATE_TO   = END_DATE.strftime("%Y-%m-%d")

print(f"Date range: {START_DATE} → {END_DATE}")

In [ ]:
# ── Authenticate & Fetch Sales Points ──────────────────────────────

from pipeline.sbis_api import authenticate, get_sales_points

sid = authenticate()
points = get_sales_points(sid)

In [ ]:
# ── Fetch Orders & Build Transactions ──────────────────────────────

from pipeline.sbis_api import fetch_orders
from pipeline.transforms import build_transactions

raw = fetch_orders(sid, points, START_DATE, END_DATE)
transactions_df = build_transactions(raw)

print(f"\n{len(transactions_df):,} transaction rows")
transactions_df.head()

In [ ]:
# ── Write to Delta Table (delete + append for date range) ─────────

from pipeline.transforms import save_delta, TRANSACTIONS_SCHEMA
from pipeline.config import TRANSACTIONS_TABLE

save_delta(transactions_df, TRANSACTIONS_TABLE, TRANSACTIONS_SCHEMA, DATE_FROM, DATE_TO)
print("Done.")